# 1 🔶 Retrieve the proximal source domain

In [ ]:
from datetime import datetime
import pyranges as pr
import pandas as pd
import numpy as np
import os


def df_to_ranges(df):
    """
    Desc:
        Convert input dataframe to pyranges format
    """
    df = df.rename(columns={'#CHROM': 'Chromosome', 'POS': 'Start'})
    df['End'] = df['Start']  # single base variant
    return pr.PyRanges(df[['Chromosome', 'Start', 'End']].copy())


def retrieve_link(df, chrom_col, pos_col, prefix=''):
    link = [f'{prefix}{chro}_{pos}' for chro, pos in zip(df[chrom_col], df[pos_col])]
    return link


def retrieve_close_samples(source_bn, source_pt, target, distance):
    """
    Desc:
        Find samples close to the target domain
    Args:
        source_bn: source domain negative samples
        source_pt: source domain positive samples
        target: target domain
        distance: select samples within distance bp of the target context
    Returns:
        source_pt_near: selected mutation sites + start distance up and down from the site (positive samples)
        source_bn_near: selected mutation sites + start distance up and down from the site (negative samples)
        source_pt_choose: selected mutation information (positive samples)
        source_bn_choose: selected mutation information (negative samples)
    """
    # Convert to PyRanges
    target_ranges = df_to_ranges(target)
    source_pt_ranges = df_to_ranges(source_pt)
    source_bn_ranges = df_to_ranges(source_bn)

    # Set ±50bp range, first add buffer to target
    target_buffered = target_ranges.slack(distance)  # slack() automatically expands Start/End left and right

    # Find overlapping regions
    pt_near = source_pt_ranges.join(target_buffered)
    bn_near = source_bn_ranges.join(target_buffered)

    # Convert back to DataFrame
    source_pt_near = pt_near.df
    source_bn_near = bn_near.df

    # Match with source data again
    source_pt_near['link'] = retrieve_link(source_pt_near, 'Chromosome', 'Start')
    source_bn_near['link'] = retrieve_link(source_bn_near, 'Chromosome', 'Start')
    source_pt['link'] = retrieve_link(source_pt, '#CHROM', 'POS')
    source_bn['link'] = retrieve_link(source_bn, '#CHROM', 'POS')
    source_pt_choose = source_pt[source_pt['link'].isin(source_pt_near['link'])]
    source_bn_choose = source_bn[source_bn['link'].isin(source_bn_near['link'])]

    # Return results
    return source_pt_near, source_bn_near, source_pt_choose, source_bn_choose


def retrieve_far_samples(
    source_bn,
    source_pt,
    target,
    save_dir,
    min_distance=2000,
    n=1000,
    random_state=913
):
    """
    Desc:
        Select mutations at least min_distance away from target
        And randomly sample n samples (positive and negative separately)

    Args:
        source_bn: source domain negative samples DataFrame
        source_pt: source domain positive samples DataFrame
        target: target domain DataFrame
        min_distance: minimum distance (default 2000bp)
        n: number of random samples per class
        random_state: random seed
    Returns:
        source_pt_far_sampled
        source_bn_far_sampled
    """
    # Convert to PyRanges
    target_ranges = df_to_ranges(target)
    source_pt_ranges = df_to_ranges(source_pt)
    source_bn_ranges = df_to_ranges(source_bn)
    # First construct target ± min_distance
    target_buffered = target_ranges.slack(min_distance)
    # Find "nearby" samples
    pt_near = source_pt_ranges.join(target_buffered)
    bn_near = source_bn_ranges.join(target_buffered)
    pt_near_df = pt_near.df
    bn_near_df = bn_near.df
    # Construct link for inverse selection
    pt_near_df["link"] = retrieve_link(pt_near_df, "Chromosome", "Start")
    bn_near_df["link"] = retrieve_link(bn_near_df, "Chromosome", "Start")
    source_pt["link"] = retrieve_link(source_pt, "#CHROM", "POS")
    source_bn["link"] = retrieve_link(source_bn, "#CHROM", "POS")
    # Inverse selection: remove samples within 2kb
    source_pt_far = source_pt[~source_pt["link"].isin(pt_near_df["link"])]
    source_bn_far = source_bn[~source_bn["link"].isin(bn_near_df["link"])]
    # Random sampling
    source_pt_far_sampled = source_pt_far.sample(n=n//2, random_state=random_state)
    source_bn_far_sampled = source_bn_far.sample(n=n//2, random_state=random_state)
    source_pt_far_sampled.to_csv(f"{save_dir}/pos.txt", sep='\t', index=False)
    source_bn_far_sampled.to_csv(f"{save_dir}/neg.txt", sep='\t', index=False)


def retrieve_based_on_granularity(source_pt_path, source_bn_path, target_path, save_dir):
    """Extract information based on different granularities"""
    print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Start screening...")
    # Source domain
    source_pt = pd.read_csv(source_pt_path, sep='\t')
    source_bn = pd.read_csv(source_bn_path, sep='\t')
    # Remove mutually exclusive samples in source domain
    pt_vids = set(source_pt['variant38'])
    bn_vids = set(source_bn['variant38'])
    source_pt = source_pt[~source_pt['variant38'].isin(bn_vids)]
    source_bn = source_bn[~source_bn['variant38'].isin(pt_vids)]
    # Target domain
    target = pd.read_csv(target_path, sep='\t')
    target_vids = set(target['variant_hg38'])
    # Remove domain mutually exclusive samples
    source_pt = source_pt[~source_pt['variant38'].isin(target_vids)]
    source_bn = source_bn[~source_bn['variant38'].isin(target_vids)]
    # Limit distance
    distance_list = list(range(500, 5500, 500))
    pos_num = []
    neg_num = []
    for distance in distance_list:
        # Get nearby mutation information
        source_pt_near, source_bn_near, source_pt_choose, source_bn_choose = retrieve_close_samples(source_bn, source_pt, target, distance)
        print(f"[INFO] Selected {distance}bp surrounding region")
        tmp_save_dir = f"{save_dir}/near_{distance}bp/"
        os.makedirs(tmp_save_dir, exist_ok=True)
        source_pt_choose.to_csv(f"{tmp_save_dir}/pos.txt", sep='\t', index=False)
        source_bn_choose.to_csv(f"{tmp_save_dir}/neg.txt", sep='\t', index=False)
        pos_num.append(len(source_pt_choose))
        neg_num.append(len(source_bn_choose))
        print(f"[INFO] Positive sample count: {len(source_pt_choose)}")
        print(f"[INFO] Negative sample count: {len(source_bn_choose)}")
    stat_df = pd.DataFrame({'Distance': distance_list, 
                            'Source_sample_amount_positive': pos_num,
                            'Source_sample_amount_negative': neg_num})
    stat_df['Source_sample_amount'] = stat_df['Source_sample_amount_positive'] + stat_df['Source_sample_amount_negative']
    stat_df.to_csv(f"{save_dir}/source_amount.info", sep='\t', index=False)
    print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Done!")

## 🔹 Missense source

### A. Close group

In [ ]:
ROOT_PATH = "./dataset/missense/"
source_pt_path = f"{ROOT_PATH}/missense.pt.51936.txt"
source_bn_path = f"{ROOT_PATH}/missense.bn.51936.txt"
target_path = "./dataset/synonymous/train_2362.txt"
save_dir = "./dataset/spatial_data/missense_spatial_dataset/"
retrieve_based_on_granularity(source_pt_path, source_bn_path, target_path, save_dir)

### B. Control group

In [ ]:
n_dict = {'500': 3964, '1000': 5424, '1500': 6763, '2000': 7981}
SEEDS = [1918, 1917, 74, 310, 913]

ROOT_PATH = "./dataset_missense/"
source_pt_path = f"{ROOT_PATH}/missense.pt.51936.txt"
source_pt = pd.read_csv(source_pt_path, sep='\t')
source_bn_path = f"{ROOT_PATH}/missense.bn.51936.txt"
source_bn = pd.read_csv(source_bn_path, sep='\t')

target_path = "./dataset/synonymous/train_2362.txt"
target = pd.read_csv(target_path, sep='\t')
save_dir = "./dataset/spatial_data/missense_spatial_dataset_control/"

for k, v in n_dict.items():
	for SEED in SEEDS:
		tmp_save_dir = f"{save_dir}/{k}bp/SEED_{SEED}/"
		os.makedirs(tmp_save_dir, exist_ok=True)
		retrieve_far_samples(
							source_bn=source_bn,
							source_pt=source_pt,
							target=target,
							save_dir=tmp_save_dir,
							min_distance=2000,
							n=v,
							random_state=SEED)

## 🔹 Noncoding source

### A. Close group

In [ ]:
ROOT_PATH = "./dataset/noncoding/"
source_pt_path = f"{ROOT_PATH}/noncoding.pt.62593.txt"
source_bn_path = f"{ROOT_PATH}/noncoding.bn.62593.txt"
target_path = "dataset/synonymous/train_2362.txt"
save_dir = "./dataset/spatial_data/noncoding_spatial_dataset/"
retrieve_based_on_granularity(source_pt_path, source_bn_path, target_path, save_dir)

### B. Control group

In [ ]:
n_dict = {'500': 11610, '1000': 15391, '1500': 18254, '2000': 21151}
SEEDS = [1918, 1917, 74, 310, 913]
ROOT_PATH = "./dataset/noncoding/"
source_pt_path = f"{ROOT_PATH}/noncoding.pt.62593.txt"
source_pt = pd.read_csv(source_pt_path, sep='\t')
source_bn_path = f"{ROOT_PATH}/noncoding.bn.62593.txt"
source_bn = pd.read_csv(source_bn_path, sep='\t')

target_path = "./dataset/synonymous/train_2362.txt"
target = pd.read_csv(target_path, sep='\t')
save_dir = "./dataset/spatial_data/noncoding_spatial_dataset_control/"

for k, v in n_dict.items():
	for SEED in SEEDS:
		tmp_save_dir = f"{save_dir}/{k}bp/SEED_{SEED}/"
		os.makedirs(tmp_save_dir, exist_ok=True)
		retrieve_far_samples(
							source_bn=source_bn,
							source_pt=source_pt,
							target=target,
							save_dir=tmp_save_dir,
							min_distance=2000,
							n=v,
							random_state=SEED)

# 2 🔶 Extract the features

## 🔹 Missense source

### A. Close group

In [ ]:
import lmdb
import numpy as np
import io
import os
import torch

def load_features_from_lmdb(lmdb_path, target_keys):
    """
    Read embeddings for specified variant keys from a single LMDB (data.mdb)
    Args:
        lmdb_path: LMDB directory path (containing data.mdb)
        target_keys: set(str)
    Returns:
        dict
    """
    results = {}
    env = lmdb.open(
        lmdb_path,
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False
    )
    with env.begin(write=False) as txn:
        for key in target_keys:
            key_bytes = key.encode("utf-8")
            value = txn.get(key_bytes)
            if value is None:
                continue
            buffer = io.BytesIO(value)
            npz = np.load(buffer)
            results[key] = {
                "calm_diff_feature": npz["calm_diff_feature"],
            }
    env.close()
    return results

def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
    variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
    return variant

feature_path = "./00-LLM-tools/F-CaLM/demo/missense/"
lmdb_neg_path = f"{feature_path}/bn/"
lmdb_pos_path = f"{feature_path}/pt/"

data_path = "./dataset/spatial_data/missense_spatial_dataset/"
spatial_regions = [str(i)+'bp' for i in range(500, 5500, 500)]

save_path = "./feature/missense/spatial_regions/"
os.makedirs(save_path, exist_ok=True)

for region in spatial_regions:

    print(f"\t[INFO] Region: {region}")

    neg_df = pd.read_csv(f"{data_path}/near_{region}/neg.txt", sep='\t')
    pos_df = pd.read_csv(f"{data_path}/near_{region}/pos.txt", sep='\t')

    neg_data = retrieve_variant(neg_df, '#CHROM', 'POS', 'REF', 'ALT')
    pos_data = retrieve_variant(pos_df, '#CHROM', 'POS', 'REF', 'ALT')

    neg_keys = set(neg_data)
    pos_keys = set(pos_data)

    pos_selected_feature = load_features_from_lmdb(lmdb_pos_path, pos_keys)
    neg_selected_feature = load_features_from_lmdb(lmdb_neg_path, neg_keys)

    if len(pos_selected_feature) == 0 and len(neg_selected_feature) == 0:
        print(f"\t[WARN] No data found for {region}, skipping.")
        continue

    save_path_tmp = f"{save_path}/{region}"
    os.makedirs(save_path_tmp, exist_ok=True)

    torch.save(pos_selected_feature, f"{save_path_tmp}/CaLM_pos.pth")
    torch.save(neg_selected_feature, f"{save_path_tmp}/CaLM_neg.pth")

    print(f"\t[SAVE] pos: {len(pos_selected_feature)} | "
          f"neg: {len(neg_selected_feature)} saved to {save_path_tmp}")

### B. Control group

In [ ]:
feature_path = "./00-LLM-tools/F-CaLM/demo/missense/"
lmdb_neg_path = f"{feature_path}/bn/"
lmdb_pos_path = f"{feature_path}/pt/"

data_path = "./dataset/spatial_data/missense_spatial_dataset_control/"
spatial_regions = [str(i)+'bp' for i in range(500, 2500, 500)]
SEEDS = [1918, 1917, 74, 310, 913]

save_path = "./feature/missense/spatial_regions_control/"
os.makedirs(save_path, exist_ok=True)

for SEED in SEEDS:
	for region in spatial_regions:
		print(f"\t[INFO] Region: {region}")
		neg_df = pd.read_csv(f"{data_path}/{region}/SEED_{SEED}/neg.txt", sep='\t')
		pos_df = pd.read_csv(f"{data_path}/{region}/SEED_{SEED}/pos.txt", sep='\t')
		print(f"[INFO] neg df: {len(neg_df)} | pos df: {len(pos_df)}")
		neg_data = retrieve_variant(neg_df, '#CHROM', 'POS', 'REF', 'ALT')
		pos_data = retrieve_variant(pos_df, '#CHROM', 'POS', 'REF', 'ALT')
		neg_keys = set(neg_data)
		pos_keys = set(pos_data)

		pos_selected_feature = load_features_from_lmdb(lmdb_pos_path, pos_keys)
		neg_selected_feature = load_features_from_lmdb(lmdb_neg_path, neg_keys)

		if len(pos_selected_feature) == 0 and len(neg_selected_feature) == 0:
			print(f"\t[WARN] No data found for {region}, skipping.")
			continue

		save_path_tmp = f"{save_path}/{region}/SEED_{SEED}/"
		os.makedirs(save_path_tmp, exist_ok=True)

		torch.save(pos_selected_feature, f"{save_path_tmp}/CaLM_pos.pth")
		torch.save(neg_selected_feature, f"{save_path_tmp}/CaLM_neg.pth")

	print(f"\t[SAVE] pos: {len(pos_selected_feature)} | "
		  f"neg: {len(neg_selected_feature)} saved to {save_path_tmp}")

## 🔹 Noncoding source

### A. Close group

In [ ]:
import os
import torch
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")


def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
	variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
	return variant

In [ ]:
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading source data...")
feature_path = "./dataset_noncoding_v2/llm/"
neg_feature = torch.load(f"{feature_path}/GPNMSA_neg.pth")
pos_feature = torch.load(f"{feature_path}/GPNMSA_pos.pth")

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Processing filter...")
data_path = "./dataset/spatial_data/noncoding_spatial_dataset/"
spatial_regions = [str(i)+'bp' for i in range(500, 5500, 500)]

save_path = "./feature/noncoding/spatial_regions/"
os.makedirs(save_path, exist_ok=True)

for region in spatial_regions:
	print(f"\t[INFO] Region: {region}")
	neg_df = pd.read_csv(f"{data_path}/near_{region}/neg.txt", sep='\t')
	pos_df = pd.read_csv(f"{data_path}/near_{region}/pos.txt", sep='\t')

	neg_data = retrieve_variant(neg_df, '#CHROM', 'POS', 'REF', 'ALT')
	pos_data = retrieve_variant(pos_df, '#CHROM', 'POS', 'REF', 'ALT')

	pos_selected_feature = {k: v for k, v in pos_feature.items() if k in pos_data}
	neg_selected_feature = {k: v for k, v in neg_feature.items() if k in neg_data}

	if len(pos_selected_feature) == 0 and len(neg_selected_feature) == 0:
		print(f"\t[WARN] No data found for level {region}, skipping.")
		continue

	save_path_tmp = f"{save_path}/{region}"
	os.makedirs(save_path_tmp, exist_ok=True)

	torch.save(pos_selected_feature, f"{save_path_tmp}/GPN_MSA_pos.pth")
	torch.save(neg_selected_feature, f"{save_path_tmp}/GPN_MSA_neg.pth")

	print(f"\t[SAVE] pos: {len(pos_selected_feature)} | neg: {len(neg_selected_feature)} saved to {save_path_tmp}")

### B. Control group

In [ ]:
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading source data...")
feature_path = "./dataset_noncoding_v2/llm/"
neg_feature = torch.load(f"{feature_path}/GPNMSA_neg.pth")
pos_feature = torch.load(f"{feature_path}/GPNMSA_pos.pth")

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Processing filter...")
data_path = "./dataset/spatial_data/noncoding_spatial_dataset_control/"
spatial_regions = [str(i)+'bp' for i in range(500, 2500, 500)]
SEEDS = [1918, 1917, 74, 310, 913]

save_path = "./feature/noncoding/spatial_regions_control/"
os.makedirs(save_path, exist_ok=True)

for region in spatial_regions:
    for SEED in SEEDS:
        print(f"\t[INFO] Region: {region}")
        neg_df = pd.read_csv(f"{data_path}/{region}/SEED_{SEED}/neg.txt", sep='\t')
        pos_df = pd.read_csv(f"{data_path}/{region}/SEED_{SEED}/pos.txt", sep='\t')

        neg_data = retrieve_variant(neg_df, '#CHROM', 'POS', 'REF', 'ALT')
        pos_data = retrieve_variant(pos_df, '#CHROM', 'POS', 'REF', 'ALT')

        pos_selected_feature = {k: v for k, v in pos_feature.items() if k in pos_data}
        neg_selected_feature = {k: v for k, v in neg_feature.items() if k in neg_data}

        if len(pos_selected_feature) == 0 and len(neg_selected_feature) == 0:
            print(f"\t[WARN] No data found for level {region}, skipping.")
            continue

        save_path_tmp = f"{save_path}/{region}/SEED_{SEED}/"
        os.makedirs(save_path_tmp, exist_ok=True)

        torch.save(pos_selected_feature, f"{save_path_tmp}/GPN_MSA_pos.pth")
        torch.save(neg_selected_feature, f"{save_path_tmp}/GPN_MSA_neg.pth")

        print(f"\t[SAVE] pos: {len(pos_selected_feature)} | neg: {len(neg_selected_feature)} saved to {save_path_tmp}")

# 3 🔶 Train the model

## 🔹 Noncoding source

In [ ]:
# torch pkgs
import torch
from torch.utils.data import DataLoader, ConcatDataset

# common pkgs
import os
import sys
import yaml
import random
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

with open('./config/config.yaml', 'r', encoding='utf-8') as file:
	config = yaml.safe_load(file)['baseline']

# setting GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Debug] Using device: {device}")
print(f"[Debug] CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"[Debug] torch.cuda.current_device() = {torch.cuda.current_device()}")
print(f"[Debug] torch.cuda.get_device_name(0) = {torch.cuda.get_device_name(0)}")

# customize functions
from functions.Dataset import MyDataset, LMDBDataset, concat_and_loader
from functions.ModelScaffold import model_fn, build_backbone
from functions.General_train import train_model
from functions.General_evaluate import test_model_detail, plot_losses_single
from functions.DyPositionTransformer import DyPositionTransformer
from functions.Utils import set_full_deterministic_seed

### A. Close group

In [ ]:
# Control random seed
SEED = 42
set_full_deterministic_seed(SEED)

# Import target domain data
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading target data...")
TARGET_ROOT_PATH = "./feature/"
tgt_test_bn = MyDataset(f"{TARGET_ROOT_PATH}/test_target_neg.pth", label=0)
tgt_test_pt = MyDataset(f"{TARGET_ROOT_PATH}/test_target_pos.pth", label=1)
tgt_test = ConcatDataset([tgt_test_bn, tgt_test_pt])
tgt_test_loader = DataLoader(tgt_test, batch_size=32, shuffle=False, num_workers=4)

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading source data...")
SOURCE_ROOT_PATH = "./feature/noncoding/spatial_regions/"
noncoding_info_df = pd.read_csv("./dataset/spatial_data/noncoding_spatial_dataset/source_amount.info", sep='\t')


for _, row in noncoding_info_df.iterrows():
    distance = row['Distance']
    pos_amount = row['Source_sample_amount_positive']
    neg_amount = row['Source_sample_amount_negative']
    # Import source domain data
    src_train_bn = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/GPN_MSA_neg_{neg_amount}.pth", label=0)
    src_train_pt = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/GPN_MSA_pos_{pos_amount}.pth", label=1)
    src_train_data = ConcatDataset([src_train_pt, src_train_bn])
    src_train_loader, src_val_loader = concat_and_loader(src_train_data, split_rate=0.8, batch_size=64)

    # Output feature information
    feature_name = 'gpn_msa' 
    # Start testing the backbone model
    
    SAVE_DIR = f"./result/section_3/spatial/noncoding/{distance}/"
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Start training
    print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Begin to train... ")
    src_path = os.path.join(SAVE_DIR, f"source_ResNet1D.pt")
    model = model_fn(backbone_name="ResNet1D", seq_len=129, embed_dim=768).to(device)
    model, _, _ = train_model(model,
                              src_train_loader, 
                              src_val_loader, 
                              config['epochs'], 
                              float(config['lr']), 
                              config['patience'],
                              save_path=src_path, 
                              device='cuda',
                              strategy_mode='plain',
                              feature_name=feature_name)
    metric_path = src_path.replace(".pt", "_metric.txt")
    save_preds_path = src_path.replace(".pt", "_pred.txt")
    test_model_detail(model=model, 
                      dataloader=tgt_test_loader, 
                      savepath=metric_path, 
                      save_preds_path=save_preds_path, 
                      device=device, 
                      feature_name=feature_name)

### B. Control group

In [ ]:
# Import target domain data
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading target data...")
TARGET_ROOT_PATH = "./feature/"
tgt_test_bn = MyDataset(f"{TARGET_ROOT_PATH}/test_target_neg.pth", label=0)
tgt_test_pt = MyDataset(f"{TARGET_ROOT_PATH}/test_target_pos.pth", label=1)
tgt_test = ConcatDataset([tgt_test_bn, tgt_test_pt])
tgt_test_loader = DataLoader(tgt_test, batch_size=32, shuffle=False, num_workers=4)

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading source data...")
SOURCE_ROOT_PATH = "./feature/noncoding/spatial_regions_control/"
noncoding_info_df = pd.read_csv("./dataset/spatial_data/noncoding_spatial_dataset_control/source_amount.info", sep='\t')
SEEDS = [1918, 1917, 74, 310, 913]

for _, row in noncoding_info_df.iterrows():
    for SEED in SEEDS:
        set_full_deterministic_seed(SEED)
        distance = row['Distance']
        pos_amount = row['Source_sample_amount_positive']
        neg_amount = row['Source_sample_amount_negative']
        # Import source domain data
        src_train_bn = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/SEED_{SEED}/GPN_MSA_neg.pth", label=0)
        src_train_pt = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/SEED_{SEED}/GPN_MSA_pos.pth", label=1)
        src_train_data = ConcatDataset([src_train_pt, src_train_bn])
        src_train_loader, src_val_loader = concat_and_loader(src_train_data, split_rate=0.8, batch_size=64, seed=SEED)

        # Output feature information
        feature_name = 'gpn_msa' 
        # Start testing the backbone model
        SAVE_DIR = f"./result/section_3/spatial/noncoding_control/{distance}/SEED_{SEED}/"
        os.makedirs(SAVE_DIR, exist_ok=True)

        # Start training
        print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Begin to train... ")
        src_path = os.path.join(SAVE_DIR, f"source_ResNet1D.pt")
        model = model_fn(backbone_name="ResNet1D", seq_len=129, embed_dim=768).to(device)
        trained_model, _, _ = train_model(model,
                                          src_train_loader, 
                                          src_val_loader, 
                                          config['epochs'], 
                                          float(config['lr']), 
                                          config['patience'],
                                          save_path=src_path, 
                                          device='cuda',
                                          mute=True,
                                          strategy_mode='plain',
                                          feature_name=feature_name)
        metric_path = src_path.replace(".pt", "_metric.txt")
        save_preds_path = src_path.replace(".pt", "_pred.txt")
        test_model_detail(model=trained_model, 
                          dataloader=tgt_test_loader, 
                          savepath=metric_path, 
                          save_preds_path=save_preds_path, 
                          device=device, 
                          feature_name=feature_name)

## 🔹 Missense source

### A. Close group

In [ ]:
# Control random seed
SEED = 913
set_full_deterministic_seed(SEED)

# Import target domain data
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading target data...")
TARGET_ROOT_PATH = "./feature/"
tgt_test_bn = MyDataset(f"{TARGET_ROOT_PATH}/test_target_neg.pth", label=0)
tgt_test_pt = MyDataset(f"{TARGET_ROOT_PATH}/test_target_pos.pth", label=1)
tgt_test = ConcatDataset([tgt_test_bn, tgt_test_pt])
tgt_test_loader = DataLoader(tgt_test, batch_size=32, shuffle=False, num_workers=4)

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading source data...")
SOURCE_ROOT_PATH = "./feature/missense/spatial_regions/"
info_df = pd.read_csv("./dataset/spatial_data/missense_spatial_dataset/source_amount.info", sep='\t')
info_df = info_df[info_df['Distance'] <= 2000]

for _, row in info_df.iterrows():
	distance = row['Distance']
	pos_amount = row['Source_sample_amount_positive']
	neg_amount = row['Source_sample_amount_negative']
	# Import source domain data
	src_train_bn = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/CaLM_neg.pth", label=0)
	src_train_pt = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/CaLM_pos.pth", label=1)
	src_train_data = ConcatDataset([src_train_pt, src_train_bn])
	src_train_loader, src_val_loader = concat_and_loader(src_train_data, split_rate=0.8, batch_size=64)

	# Output feature information
	feature_name = 'calm_diff_feature'
	# Start testing the backbone model
	SAVE_DIR = f"./result/section_3/spatial/missense/{distance}/"
	os.makedirs(SAVE_DIR, exist_ok=True)

	# Start training
	print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Begin to train... ")
	src_path = os.path.join(SAVE_DIR, f"source_positionTransformer.pt")
	my_model = DyPositionTransformer(
				input_dim=768,
				num_layers=4,
				d_model=256,
				num_heads=4,
				dff=512,
				dropout_rate=0.1,
				head_hidden=128,
		).to(device)  
	model, _, _ = train_model(my_model,
							  src_train_loader, 
							  src_val_loader, 
							  config['epochs'], 
							  float(config['lr']), 
							  config['patience'],
							  save_path=src_path, 
							  device='cuda',
							  strategy_mode='plain',
							  feature_name=feature_name)
	metric_path = src_path.replace(".pt", "_metric.txt")
	save_preds_path = src_path.replace(".pt", "_pred.txt")
	test_model_detail(model=model, 
					dataloader=tgt_test_loader, 
					savepath=metric_path, 
					save_preds_path=save_preds_path, 
					device=device, 
					feature_name=feature_name)

### B. Control group

In [ ]:
# 导入目标域数据
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading target data...")
TARGET_ROOT_PATH = "./feature/"
tgt_test_bn = MyDataset(f"{TARGET_ROOT_PATH}/test_target_neg.pth", label=0)
tgt_test_pt = MyDataset(f"{TARGET_ROOT_PATH}/test_target_pos.pth", label=1)
tgt_test = ConcatDataset([tgt_test_bn, tgt_test_pt])
tgt_test_loader = DataLoader(tgt_test, batch_size=32, shuffle=False, num_workers=4)

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Loading source data...")
SOURCE_ROOT_PATH = "./feature/missense/spatial_regions_control/"
SEEDS = [1918, 1917, 74, 310, 913]
distance_list = list(range(500, 2500, 500))

for distance in distance_list:
    for SEED in SEEDS:
        # Import source domain data
        src_train_bn = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/SEED_{SEED}/CaLM_neg.pth", label=0)
        src_train_pt = MyDataset(f"{SOURCE_ROOT_PATH}/{distance}bp/SEED_{SEED}/CaLM_pos.pth", label=1)
        src_train_data = ConcatDataset([src_train_pt, src_train_bn])
        src_train_loader, src_val_loader = concat_and_loader(src_train_data, split_rate=0.8, batch_size=64, seed=SEED)

        # Output feature information
        feature_name = 'calm_diff_feature' 
        # Start testing the backbone model
        SAVE_DIR = f"./result/section_3/spatial/missense_control/{distance}/SEED_{SEED}/"
        os.makedirs(SAVE_DIR, exist_ok=True)

        # 开始训练
        print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Begin to train... ")
        src_path = os.path.join(SAVE_DIR, f"source_positionTransformer.pt")
        my_model = DyPositionTransformer(
                                         input_dim=768,
                                         num_layers=4,
                                         d_model=256,
                                         num_heads=4,
                                         dff=512,
                                         dropout_rate=0.1,
                                         head_hidden=128,
                                        ).to(device) 
        trained_model, _, _ = train_model(model,
                                          src_train_loader, 
                                          src_val_loader, 
                                          config['epochs'], 
                                          float(config['lr']), 
                                          config['patience'],
                                          save_path=src_path, 
                                          device='cuda',
                                          mute=True,
                                          strategy_mode='plain',
                                          feature_name=feature_name)
        metric_path = src_path.replace(".pt", "_metric.txt")
        save_preds_path = src_path.replace(".pt", "_pred.txt")
        test_model_detail(model=trained_model, 
                          dataloader=tgt_test_loader, 
                          savepath=metric_path, 
                          save_preds_path=save_preds_path, 
                          device=device, 
                          feature_name=feature_name)